In [1]:
import pandas as pd
import sys
sys.path.append('/home/azureuser/cloudfiles/code/Users/manhductranvu/reeval-multi/mirt-official') 
from load_params import load_and_rotate

resmat = pd.read_pickle("../data/resmat.pkl")

theta, a, b = load_and_rotate()

# Step 1: Get the final theta tensor into a NumPy array
theta_abilities = theta
# Step 2: Create a labeled pandas DataFrame
# Use the model names from your original resmat for the index
model_names = resmat.index
factor_names = [f'F{i+1}' for i in range(theta.shape[1])]
ability_df = pd.DataFrame(theta_abilities, index=model_names, columns=factor_names)

/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu10c/code/Users/manhductranvu/reeval-multi/mirt-official/load_params.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m

--- Initial Loaded Data ---
Original theta shape: torch.Size([183, 6])
Original 'a' matrix shape: torch.Size([78712, 6])

--- After Rotation of 'a' ---
Rotated 'a' matrix shape: (78712, 6)

--- After Transformation of 'theta' ---
Transformed theta shape: (183, 6)

--- Final Standardized Z-Scores (from transformed theta) ---
These are the scores you should use for interpretation.
[[ 1.5905465   3.49991235 -2.70676842  0.80146618  0.72562385 -0.62409166]
 [ 1.26710786 -0.50314049  1.10198573 -1.1741239   0.34140497  0.07110775]
 [ 1.86002922  0.60271614  0.90773613  0.07469808 -0.76376825 -1.55688678]
 ...
 [-1.05323766 -0.53844968 -2.76657604  1.42263159  0.66665258  2.97106778]
 [-0.43951293 -0.54343183 -2.60149674  2.23806708  0.32196824  3.12436554]
 [ 0.04734164 -0.47165056 -1.30415358  1.18832103  1.00318341  2.02381296]]
--- After Rotation of 'a' ---
Rotated 'a' matrix shape: (78712, 6)

--- After Transformation of 'theta' ---
Transformed theta shape: (183, 6)

--- Final Standardi

In [35]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

def xgreg(X, y, test_size=0.2, top=15, save_path=None, pattern=None):
    """
    Trains an XGBoost Regressor, evaluates it, and displays performance metrics.

    Includes a train-test split to provide a realistic evaluation of the model's
    performance on unseen data, helping to identify overfitting.

    Args:
        X (pd.DataFrame): Feature data.
        y (pd.Series): Target data.
        test_size (float, optional): The proportion of the dataset to include in the
                                     test split. If 0, trains on the full dataset.
                                     Defaults to 0.2.
        top (int, optional): Number of top models to display in the comparison.
                             Defaults to 15.
        save_path (str, optional): Path to save the comparison CSV file.
                                   Defaults to None.
        pattern (str, optional): A string pattern to filter models by name for
                                 the comparison table. Defaults to None.

    Returns:
        XGBRegressor: The trained XGBoost model instance.
    """
    X = X.copy()

    # --- Step 1: Split data into training and testing sets ---
    # This is the most important change to prevent overfitting evaluation.
    if test_size > 0 and test_size < 1:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        print(f"--- Data split into {len(X_train)} training and {len(X_test)} test samples ---\n")
    else:
        # If test_size is 0, use the full dataset for training (original behavior)
        X_train, y_train = X, y
        X_test, y_test = X, y # Test set is the same as train set for this case
        print("--- No train-test split. Training and evaluating on the full dataset. ---\n")

    # Step 2: Initialize and train the XGBoost Regressor on the TRAINING data
    reg = XGBRegressor(
        n_estimators=200,
        learning_rate=0.005,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    reg.fit(X_train, y_train)

    # --- Step 3: Evaluate Performance on BOTH Training and Test Sets ---
    y_pred_train = reg.predict(X_train)
    y_pred_test = reg.predict(X_test)

    print("--- Model Performance Metrics ---")
    print(f"Training Set R-squared:   {r2_score(y_train, y_pred_train):.4f}")
    # A large gap between train and test R-squared indicates overfitting
    print(f"Test Set R-squared:       {r2_score(y_test, y_pred_test):.4f}\n")

    print(f"Training Set MAE:         {mean_absolute_error(y_train, y_pred_train):.4f}")
    print(f"Test Set MAE:             {mean_absolute_error(y_test, y_pred_test):.4f}\n")
    
    # --- Feature Importances and Correlations (from the trained model) ---
    importances = reg.feature_importances_
    print("--- Data-Driven Feature Importances for the Metric ---")
    for factor, importance in zip(X.columns, importances):
        print(f"{factor}: {importance:.4f}")

    print("\n--- Individual Factor Correlations with Performance (Full Dataset) ---")
    for factor in X.columns:
        correlation = X[factor].corr(y)
        print(f"{factor}: {correlation:.4f}")

    # --- Step 4: Detailed Comparison on the TEST Set ---
    # This shows the model's true performance on unseen data.
    print("\n=== Comparison: Regression Predictions vs Actual (on TEST SET) ===\n")

    comparison_df = pd.DataFrame({
        'Actual_Mean': y_test,
        'Regression_Pred': y_pred_test,
        'Difference': y_pred_test - y_test,
        'Abs_Difference': np.abs(y_pred_test - y_test)
    })

    if pattern:
        comparison_df = comparison_df[comparison_df.index.str.contains(pattern, case=False)]

    comparison_df = comparison_df.sort_values('Actual_Mean', ascending=False).round(3)

    # Logic for printing top N models
    num_to_display = top if top is not None and top <= len(comparison_df) else len(comparison_df)
    if top:
         print(f"Top {num_to_display} models comparison (Test Set):")
    print(comparison_df.head(num_to_display))

    if save_path:
        comparison_df.to_csv(save_path)
        print(f"\nTest set comparison saved to {save_path}")

    # Overall statistics for the TEST SET
    print(f"\nOverall Test Set Statistics:")
    print(f"Mean Absolute Error: {comparison_df['Abs_Difference'].mean():.3f}")
    print(f"Root Mean Square Error: {np.sqrt((comparison_df['Difference']**2).mean()):.3f}")
    print(f"Correlation: {comparison_df['Actual_Mean'].corr(comparison_df['Regression_Pred']):.3f}")

    # Show models with largest prediction errors on the TEST SET
    print(f"\nModels with largest prediction errors (Test Set):")
    worst_predictions = comparison_df.nlargest(5, 'Abs_Difference')[['Actual_Mean', 'Regression_Pred', 'Difference']]
    print(worst_predictions)

    return reg

In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

def ridgereg(X, y, test_size=0.2, top=15, save_path=None, pattern=None):
    """
    Trains a Ridge Regression model, evaluates it, and displays performance metrics.

    Includes a train-test split to provide a realistic evaluation of the model's
    performance on unseen data, helping to identify overfitting.

    Args:
        X (pd.DataFrame): Feature data.
        y (pd.Series): Target data.
        test_size (float, optional): The proportion of the dataset to include in the
                                     test split. If 0, trains on the full dataset.
        top (int, optional): Number of top models to display in the comparison.
                             Defaults to 15.
        save_path (str, optional): Path to save the comparison CSV file.
                                   Defaults to None.
        pattern (str, optional): A string pattern to filter models by name for
                                 the comparison table. Defaults to None.

    Returns:
        RidgeCV: The trained Ridge model instance.
    """
    X = X.copy()

    # --- Step 1: Split data into training and testing sets ---
    if test_size > 0 and test_size < 1:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        print(f"--- Data split into {len(X_train)} training and {len(X_test)} test samples ---\n")
    else:
        X_train, y_train = X, y
        X_test, y_test = X, y
        print("--- No train-test split. Training and evaluating on the full dataset. ---\n")

    # --- Step 2: Train Ridge Regression with CV ---
    alphas = np.logspace(-3, 3, 50)  # range of possible regularization strengths
    reg = RidgeCV(alphas=alphas)
    reg.fit(X_train, y_train)

    # --- Step 3: Evaluate Performance on BOTH Training and Test Sets ---
    y_pred_train = reg.predict(X_train)
    y_pred_test = reg.predict(X_test)

    print("--- Model Performance Metrics ---")
    print(f"Training Set R-squared:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"Test Set R-squared:       {r2_score(y_test, y_pred_test):.4f}\n")

    print(f"Training Set MAE:         {mean_absolute_error(y_train, y_pred_train):.4f}")
    print(f"Test Set MAE:             {mean_absolute_error(y_test, y_pred_test):.4f}\n")

    # --- Step 4: Coefficients and Correlations ---
    coefs = pd.Series(reg.coef_, index=X.columns)
    print("--- Ridge Regression Coefficients (Factor Weights) ---")
    print(coefs.round(4).to_string())

    print("\n--- Individual Factor Correlations with Performance (Full Dataset) ---")
    for factor in X.columns:
        correlation = X[factor].corr(y)
        print(f"{factor}: {correlation:.4f}")

    # --- Step 5: Detailed Comparison on the TEST Set ---
    print("\n=== Comparison: Regression Predictions vs Actual (on TEST SET) ===\n")

    comparison_df = pd.DataFrame({
        'Actual_Mean': y_test,
        'Regression_Pred': y_pred_test,
        'Difference': y_pred_test - y_test,
        'Abs_Difference': np.abs(y_pred_test - y_test)
    }, index=X_test.index)

    if pattern:
        comparison_df = comparison_df[comparison_df.index.str.contains(pattern, case=False)]

    comparison_df = comparison_df.sort_values('Actual_Mean', ascending=False).round(3)

    num_to_display = top if top and top <= len(comparison_df) else len(comparison_df)
    if top:
         print(f"Top {num_to_display} models comparison (Test Set):")
    print(comparison_df.head(num_to_display))

    if save_path:
        comparison_df.to_csv(save_path)
        print(f"\nTest set comparison saved to {save_path}")

    # Overall statistics for the TEST SET
    print(f"\nOverall Test Set Statistics:")
    print(f"Mean Absolute Error: {comparison_df['Abs_Difference'].mean():.3f}")
    print(f"Root Mean Square Error: {np.sqrt((comparison_df['Difference']**2).mean()):.3f}")
    print(f"Correlation: {comparison_df['Actual_Mean'].corr(comparison_df['Regression_Pred']):.3f}")

    print(f"\nModels with largest prediction errors (Test Set):")
    worst_predictions = comparison_df.nlargest(5, 'Abs_Difference')[['Actual_Mean', 'Regression_Pred', 'Difference']]
    print(worst_predictions)

    return reg


In [9]:
df_theta = ability_df
df_performance = pd.read_csv('../data/scenario_probs.csv')  # (models x scenarios) DataFrame of raw scores

# Step 1: Create the "Ground Truth" Score (your target variable 'y')
scenarios = ['med_qa']
ground_truth_scores = pd.DataFrame(df_performance[scenarios].mean(axis=1).to_list(), index=df_theta.index.to_list(), columns=['score'])
reg = ridgereg(df_theta, ground_truth_scores['score'], test_size=0.2, top=15, save_path=None, pattern="anthropic/cl|openai/gp|google/ge|meta/ll")

--- Data split into 146 training and 37 test samples ---

--- Model Performance Metrics ---
Training Set R-squared:   0.5815
Test Set R-squared:       0.2163

Training Set MAE:         0.1741
Test Set MAE:             0.2180

--- Ridge Regression Coefficients (Factor Weights) ---
F1   -0.1955
F2    0.0362
F3   -0.0209
F4    0.0504
F5   -0.0054
F6    0.0118

--- Individual Factor Correlations with Performance (Full Dataset) ---
F1: -0.6920
F2: 0.0294
F3: -0.2505
F4: 0.5249
F5: 0.0102
F6: 0.2176

=== Comparison: Regression Predictions vs Actual (on TEST SET) ===

Top 11 models comparison (Test Set):
                                          Actual_Mean  Regression_Pred  \
request.model                                                            
meta/llama-3.1-405b-instruct-turbo              0.855            0.664   
anthropic/claude-3-sonnet-20240229              0.691            0.489   
anthropic/claude-2.0                            0.630            0.433   
anthropic/claude-v1.3    

In [4]:
import pandas as pd

codingarena = pd.read_csv("../data/coding-llm-arena.csv")
resmat = pd.read_pickle("../data/resmat.pkl")
models = resmat.index.to_list()
models = [m.split('/', 1)[-1] if '/' in m else m for m in models]

In [5]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0.3, 1))
codingarena['score_scaled'] = scaler.fit_transform(codingarena[['Score']])

In [6]:
import pandas as pd
from thefuzz import process, fuzz

# 1. Get the unique list of model names
choices = codingarena['Model'].unique().tolist()

# 2. Set a stricter cutoff score
cutoff_score = 95

# 3. Find the best matches and ensure 1-to-1 mapping
fuzzy_matches = {}
used_choices = set()  # Track which choices have been used

# First, get all potential matches with their scores
potential_matches = []
for model in models:
    result = process.extractOne(
        model, 
        choices, 
        scorer=fuzz.partial_ratio,
        score_cutoff=cutoff_score
    )
    if result:
        potential_matches.append((model, result[0], result[1]))  # (model, matched_choice, score)

# Sort by score (descending) to prioritize better matches
potential_matches.sort(key=lambda x: x[2], reverse=True)

# Assign matches ensuring no duplicates
for model, choice, score in potential_matches:
    if choice not in used_choices and model not in fuzzy_matches:
        fuzzy_matches[model] = choice
        used_choices.add(choice)
    elif model not in fuzzy_matches:
        # If the best choice is taken, try to find the next best available choice
        available_choices = [c for c in choices if c not in used_choices]
        if available_choices:
            backup_result = process.extractOne(
                model,
                available_choices,
                scorer=fuzz.partial_ratio,
                score_cutoff=cutoff_score
            )
            if backup_result:
                fuzzy_matches[model] = backup_result[0]
                used_choices.add(backup_result[0])
            else:
                fuzzy_matches[model] = None
        else:
            fuzzy_matches[model] = None

# 4. Create the mapping Series
match_map = pd.Series(fuzzy_matches)

# 5. Verify no duplicates
successful_matches = match_map.dropna()
duplicate_check = successful_matches.value_counts()
duplicates = duplicate_check[duplicate_check > 1]

print("✅ Successful Matches (1-to-1 mapping):")
print(successful_matches)
print(f"\nTotal successful matches: {len(successful_matches)}")

if len(duplicates) > 0:
    print("⚠️  Warning: Found duplicate matches:")
    print(duplicates)
else:
    print("✅ Confirmed: No duplicate matches found!")

print(f"\nUnmatched models: {len(match_map) - len(successful_matches)}")

✅ Successful Matches (1-to-1 mapping):
olmo-7b                                    olmo-7b-instruct
mixtral-8x7b-instruct-v0.1       mixtral-8x7b-instruct-v0.1
qwen1.5-7b-chat                             qwen1.5-7b-chat
mixtral-8x22b-instruct-v0.1     mixtral-8x22b-instruct-v0.1
o1-mini-2024-09-12                                  o1-mini
llama-13b                                         llama-13b
qwen1.5-72b-chat                           qwen1.5-72b-chat
command                                   command-a-03-2025
vicuna-13b-v1.3                                  vicuna-13b
mpt-30b                                        mpt-30b-chat
vicuna-7b-v1.3                                    vicuna-7b
llama-2-13b                                llama-2-13b-chat
deepseek-r1                                deepseek-r1-0528
llama-3.1-8b-instruct-turbo           llama-3.1-8b-instruct
gpt-3.5-turbo-1106                       gpt-3.5-turbo-1106
command-r                            command-r-plus-08-2024
m

In [7]:
faulty_matches = [
    "command",           # Matched to a specific dated version 'command-a-03-2025'
    "command-r",         # Matched to 'command-r-plus', a different, more capable model
    "glm",               # Matched to 'glm-4.5', a major version difference
    "gemma-2-9b-it",     # Matched to 'gemma-2-9b-it-simpo', a different fine-tuning variant
    "gemma-2-27b",       # Matched to 'gemma-2-2b-it', a completely different parameter size (27B vs 2B)
    "deepseek-v3",       # Matched to 'deepseek-v3.1', a distinct minor version update
]

# First, handle any missing values as you did before
match_map = match_map.dropna()

# Now, drop the rows corresponding to the faulty matches
match_map_cleaned = match_map.drop(labels=faulty_matches, errors='ignore')

# You can now work with the cleaned map
match_map_cleaned

olmo-7b                                    olmo-7b-instruct
mixtral-8x7b-instruct-v0.1       mixtral-8x7b-instruct-v0.1
qwen1.5-7b-chat                             qwen1.5-7b-chat
mixtral-8x22b-instruct-v0.1     mixtral-8x22b-instruct-v0.1
o1-mini-2024-09-12                                  o1-mini
llama-13b                                         llama-13b
qwen1.5-72b-chat                           qwen1.5-72b-chat
vicuna-13b-v1.3                                  vicuna-13b
mpt-30b                                        mpt-30b-chat
vicuna-7b-v1.3                                    vicuna-7b
llama-2-13b                                llama-2-13b-chat
deepseek-r1                                deepseek-r1-0528
llama-3.1-8b-instruct-turbo           llama-3.1-8b-instruct
gpt-3.5-turbo-1106                       gpt-3.5-turbo-1106
mistral-medium-2312                          mistral-medium
yi-34b-chat                                     yi-34b-chat
command-r-plus                          

In [8]:
# Create a reverse mapping from codingarena models to your resmat models
reverse_match_map = {v: k for k, v in match_map.dropna().items()}

# Add the matched model names as a new column in codingarena
codingarena_with_matches = codingarena[['Model', 'Score', 'score_scaled']].copy()
codingarena_with_matches['matched_resmat_model'] = codingarena_with_matches['Model'].map(reverse_match_map)

# Keep only rows that have successful matches
codingarena_matched = codingarena_with_matches.dropna(subset=['matched_resmat_model'])

print(f"Original codingarena rows: {len(codingarena)}")
print(f"Rows with successful matches: {len(codingarena_matched)}")
print(f"Success rate: {len(codingarena_matched)/len(codingarena)*100:.1f}%")

print("\nFirst few rows with matches:")

codingarena_matched = codingarena_matched[['matched_resmat_model', 'score_scaled']].rename(columns={
    'matched_resmat_model': 'model',
    'score_scaled': 'score'
}).set_index('model')[['score']]

codingarena_matched.head(10)

Original codingarena rows: 225
Rows with successful matches: 58
Success rate: 25.8%

First few rows with matches:


,score
model,
deepseek-r1,0.948068
deepseek-v3,0.944822
glm,0.944822
o1-2024-12-17,0.914529
o3-mini-2025-01-31,0.888563
o1-mini-2024-09-12,0.869088
command,0.864760
claude-3-5-haiku-20241022,0.853941
gpt-4o-2024-05-13,0.845286


In [9]:
df_theta = ability_df

In [10]:
models = ability_df.index.to_list()
models = [m.split('/', 1)[-1] if '/' in m else m for m in models]

In [11]:
df_theta['models'] = models
df_theta.set_index('models', inplace=True)
df_theta = df_theta[df_theta.index.isin(codingarena_matched.index)]

In [12]:
codingarena_scores_series = codingarena_matched['score']

reg = xgreg(df_theta, codingarena_scores_series, test_size=0.2, top=None, save_path='../result/codingarena_regression.csv', pattern=None)

--- Data split into 46 training and 12 test samples ---

--- Model Performance Metrics ---
Training Set R-squared:   1.0000
Test Set R-squared:       0.7716

Training Set MAE:         0.0005
Test Set MAE:             0.0420

--- Data-Driven Feature Importances for the Metric ---
F1: 0.1350
F2: 0.0381
F3: 0.3377
F4: 0.2368
F5: 0.0266
F6: 0.2258

--- Individual Factor Correlations with Performance (Full Dataset) ---
F1: -0.2947
F2: 0.1566
F3: -0.4908
F4: 0.6094
F5: -0.2353
F6: 0.1492

=== Comparison: Regression Predictions vs Actual (on TEST SET) ===

                        Actual_Mean  Regression_Pred  Difference  \
model                                                              
deepseek-r1                   0.948            0.856      -0.092   
o1-2024-12-17                 0.915            0.849      -0.066   
o1-mini-2024-09-12            0.869            0.850      -0.019   
claude-3-opus-20240229        0.824            0.784      -0.040   
gemini-1.5-pro-001            0.824 

# Math score

In [51]:
import numpy as np
import pandas as pd

def compute_math_ability(theta_df, a_df, b_series, math_items, adjust_difficulty=True):
    """
    Compute a math-specific ability score from MIRT factors, discriminations, and difficulties.
    
    Args:
        theta_df (pd.DataFrame): Factor scores per test taker (rows = test takers, cols = factors).
        a_df (pd.DataFrame): Discrimination parameters (rows = items, cols = factors).
        b_series (pd.Series): Difficulty parameters (indexed same as a_df rows).
        math_items (list): List of item IDs (subset of a_df rows) that are math items.
        adjust_difficulty (bool): Whether to weight discriminations by difficulty.
    
    Returns:
        pd.Series: Math ability scores for each test taker.
    """
    # 1. Extract discrimination vectors for math items
    A_math = a_df.loc[math_items].copy()
    b_math = b_series.loc[math_items]

    # 2. Optionally adjust by difficulty
    if adjust_difficulty:
        weights = 1 / (1 + np.abs(b_math.values))
        A_math = A_math.multiply(weights, axis=0)

    # 3. Compute math direction vector in factor space
    v_math = A_math.mean(axis=0).values  # vector of length = num_factors
    v_math = v_math / np.linalg.norm(v_math)  # normalize for stability

    # 4. Project each test taker's theta onto v_math
    theta_matrix = theta_df.values  # shape (n_test_takers, n_factors)
    math_scores = theta_matrix.dot(v_math)

    return pd.Series(math_scores, index=theta_df.index, name="Specific Ability")


In [52]:
# 
models = resmat.index.to_list()
questions = resmat.columns.get_level_values('input.text').to_list()
item_mask = resmat.columns.get_level_values('input.text') == "The __________ was a huge marketplace of Dark Web specifically famous for selling of illegal drugs & narcotics as well as you can find a wide range of other goods for sale."
the_question = resmat.columns[item_mask]

In [53]:
import json

# Extract all input.text values from resmat columns (similar to your approach)
models = resmat.index.to_list()
questions = resmat.columns.get_level_values('input.text').to_list()

# Get unique questions to avoid duplicates
unique_questions = set(questions)

# Load MMLU questions from JSON file

with open('mmlu-questions.json', 'r') as f:
	mmlu_data = json.load(f)

# Extract question texts from the loaded data
question_texts = set([item['question'] for item in mmlu_data])

common_questions = list(unique_questions.intersection(question_texts))

In [54]:
models = resmat.index.to_list()
questions = resmat.columns.get_level_values('input.text').to_list()
math_items_mask = resmat.columns.get_level_values('scenario').isin(['gsm', 'math'])

math_questions = resmat.columns[math_items_mask].get_level_values('input.text').to_list()

theta_df = pd.DataFrame(theta, index=models)

a_df = pd.DataFrame(a, index=questions)

b_series = pd.Series(b, index=questions)

math_items = common_questions

# Compute math ability
math_scores = compute_math_ability(theta_df, a_df, b_series, math_items)

print("Math scores computed successfully!")
print(f"Math scores range: {math_scores.min():.3f} to {math_scores.max():.3f}")
print(f"\nTop 10 models by math ability:")
with pd.option_context('display.max_rows', None):
    display(math_scores.sort_values(ascending=False).head(100)) # Display only 10 rows

Math scores computed successfully!
Math scores range: -2.459 to 2.268

Top 10 models by math ability:


openai/gpt-4o-2024-08-06                           2.268002
openai/gpt-4o-2024-05-13                           2.262340
openai/gpt-4-turbo-2024-04-09                      2.100079
anthropic/claude-3-5-sonnet-20240620               1.957571
qwen/qwen2.5-72b-instruct-turbo                    1.916927
openai/gpt-4-0613                                  1.888635
meta/llama-3.1-405b-instruct-turbo                 1.771282
meta/llama-3.2-90b-vision-instruct-turbo           1.731077
qwen/qwen2-72b-instruct                            1.715350
writer/palmyra-x-004                               1.702725
amazon/nova-pro-v1:0                               1.644107
anthropic/claude-3-opus-20240229                   1.637535
openai/gpt-4o-mini-2024-07-18                      1.613829
deepseek-ai/deepseek-v3                            1.612869
writer/palmyra-x-v3                                1.582992
meta/llama-3.3-70b-instruct-turbo                  1.571133
google/gemini-1.5-flash-preview-0514    

In [ ]:
pd.reset_option('display.max_rows')